# vla-hands · Medium: All Appendages + Benchmark

**Assumes**: `01_intro.ipynb` — you should be comfortable with `VLAGraft`, `VisionBridge`,
`TrainingCurriculum`, and `CurriculumConfig`.

This notebook covers:

| Section | Topic |
|---------|-------|
| 2 | All 5 appendage types — table + code |
| 3 | Train each graft on a matched environment |
| 4 | Training curves (all 5 side by side) |
| 5 | Benchmark vs expert baseline |
| 6 | GIF export — expert and trained rollouts |
| 7 | What's next |

**Runtime**: ~15–25 min on a T4 GPU with early stopping enabled.

## 1. Setup

In [ ]:
!pip install -q git+https://github.com/jerod92/project-h.git@claude/vla-robotic-hands-platform-kGiza

In [ ]:
import torch
import matplotlib.pyplot as plt
from IPython.display import display, Image as IPImage

from transformers import AutoProcessor, AutoModelForVision2Seq

from vla_hands.grafting.core import (
    VLAGraft,
    VisionBridge,
    GraftConfig,
    JoystickAppendage,
    DPadAppendage,
    ButtonAppendage,
    MultiButtonAppendage,
    TouchscreenAppendage,
)
from vla_hands.training.curriculum import (
    TrainingCurriculum,
    CurriculumConfig,
    QUICK_CURRICULUM,
)
from vla_hands.envs import (
    TargetNavEnvironment,
    GridWorldEnvironment,
    ButtonPressEnvironment,
    MCQButtonEnvironment,
    PointingEnvironment,
)
from vla_hands.utils.viz import plot_training_curves
from vla_hands.utils.benchmark import BenchmarkSuite
from vla_hands.utils.gif import record_expert_gif, save_rollout_gif
from vla_hands.training.expert import run_expert_baseline

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {device}")

In [ ]:
MODEL_ID = "HuggingFaceTB/SmolVLM-256M-Instruct"

processor = AutoProcessor.from_pretrained(MODEL_ID)
vlm = AutoModelForVision2Seq.from_pretrained(MODEL_ID, torch_dtype=torch.float32)
vlm = vlm.to(device)
vlm.eval()

hidden_dim = vlm.config.hidden_size
vision_dim = VLAGraft.detect_vision_dim(vlm)

print(f"hidden_dim : {hidden_dim}")
print(f"vision_dim : {vision_dim}")

## 2. All Five Appendage Types

Each appendage targets a different action space:

| Appendage | Output space | Example task |
|---|---|---|
| `JoystickAppendage` | Continuous `(dx, dy)` in `[-1, 1]²` | Navigate to a target |
| `DPadAppendage` | Discrete 5-way (up / down / left / right / stay) | Grid-world navigation |
| `ButtonAppendage` | Binary press / no-press | Trigger a single button |
| `MultiButtonAppendage` | `n` independent binary signals | Multiple-choice panel (n=4) |
| `TouchscreenAppendage` | Absolute `(x, y)` in `[0, 1]²` | Tap a target on screen |

### Why `VisionBridge`?

`VisionBridge` wraps any appendage with a **skip connection** from the vision encoder.
Instead of only receiving the last-token hidden state from the frozen LLM decoder, the action
head also receives the mean-pooled patch embeddings directly from the vision encoder.
This bypasses the information bottleneck of the frozen decoder and dramatically speeds up
convergence on image-driven tasks.

Rule of thumb: **always use `VisionBridge` for image input tasks.**

In [ ]:
def make_graft(AppendageClass, vision_dim=vision_dim, hidden_dim=hidden_dim, **kwargs):
    """Instantiate an appendage, wrap it in VisionBridge, pair with the frozen VLM."""
    appendage = AppendageClass(hidden_dim=hidden_dim, **kwargs)
    bridged = VisionBridge(appendage, vision_dim=vision_dim)
    return VLAGraft(vlm, bridged, config=GraftConfig(feature_extraction="last"))


graft_joystick    = make_graft(JoystickAppendage)
graft_dpad        = make_graft(DPadAppendage)
graft_button      = make_graft(ButtonAppendage)
graft_mcq         = make_graft(MultiButtonAppendage, n=4)
graft_touchscreen = make_graft(TouchscreenAppendage)

all_grafts = {
    "joystick":    graft_joystick,
    "dpad":        graft_dpad,
    "button":      graft_button,
    "mcq":         graft_mcq,
    "touchscreen": graft_touchscreen,
}

for name, g in all_grafts.items():
    n_params = sum(p.numel() for p in g.parameters() if p.requires_grad)
    print(f"{name:>12s}  trainable params: {n_params:,}")

## 3. Train All Grafts

The `quick_train` helper wraps `TrainingCurriculum` with sensible defaults.
Early stopping keeps wall-clock time reasonable:

- **BC phase** stops once loss drops below `0.03`
- **RL phase** stops once episode success rate reaches `0.9`

Each appendage type is paired with the most natural environment for it.

In [ ]:
def quick_train(graft, env, name, bc_steps=300, rl_steps=50):
    """Run BC -> RL curriculum and return the metrics dict."""
    config = CurriculumConfig(
        bc_steps=bc_steps,
        rl_steps=rl_steps,
        bc_early_stop_loss=0.03,
        rl_early_stop_success=0.9,
        appendage_lr=3e-4,
        device=device,
        save_dir=f"checkpoints/{name}",
        freezing_stages=QUICK_CURRICULUM,
        eval_every=25,
        log_every=10,
        eval_episodes=5,
    )
    curriculum = TrainingCurriculum(graft, processor, env, config)
    metrics = curriculum.run()
    print(
        f"[{name}] "
        f"BC final loss: {metrics['bc']['loss'][-1]:.4f}  "
        f"RL final success: {metrics['rl']['success_rate'][-1]:.2f}"
    )
    return metrics

In [ ]:
# Joystick -> continuous navigation to a moving target
env_joystick = TargetNavEnvironment()
metrics_joystick = quick_train(graft_joystick, env_joystick, "joystick")

In [ ]:
# D-Pad -> discrete step-based grid navigation
env_dpad = GridWorldEnvironment()
metrics_dpad = quick_train(graft_dpad, env_dpad, "dpad")

In [ ]:
# Button -> single binary press when a colour matches
env_button = ButtonPressEnvironment()
metrics_button = quick_train(graft_button, env_button, "button")

In [ ]:
# MultiButton (MCQ) -> pick one of four labelled options
env_mcq = MCQButtonEnvironment()
metrics_mcq = quick_train(graft_mcq, env_mcq, "mcq")

In [ ]:
# Touchscreen -> tap the highlighted target
env_touchscreen = PointingEnvironment()
metrics_touchscreen = quick_train(graft_touchscreen, env_touchscreen, "touchscreen")

## 4. Training Curves

`plot_training_curves` returns a `matplotlib` figure. We tile all five side by side
to make it easy to compare convergence speed across appendage types.

In [ ]:
all_metrics = {
    "joystick":    metrics_joystick,
    "dpad":        metrics_dpad,
    "button":      metrics_button,
    "mcq":         metrics_mcq,
    "touchscreen": metrics_touchscreen,
}

fig, axes = plt.subplots(2, 5, figsize=(22, 8))
fig.suptitle("Training curves — all five appendage types", fontsize=14)

for col, (name, metrics) in enumerate(all_metrics.items()):
    # plot_training_curves produces its own figure; we copy the line data into our grid
    curve_fig = plot_training_curves(metrics, title=name)
    for row, src_ax in enumerate(curve_fig.axes[:2]):
        dst_ax = axes[row, col]
        for line in src_ax.lines:
            dst_ax.plot(line.get_xdata(), line.get_ydata())
        dst_ax.set_title(src_ax.get_title() or name)
        dst_ax.set_xlabel(src_ax.get_xlabel())
        dst_ax.set_ylabel(src_ax.get_ylabel())
        dst_ax.grid(alpha=0.3)
    plt.close(curve_fig)

plt.tight_layout()
plt.show()

## 5. Benchmark

`BenchmarkSuite` runs the trained graft for 20 episodes and reports `success_rate` and
`mean_reward`. We compare each graft against the scripted expert baseline.

In [ ]:
envs = {
    "joystick":    env_joystick,
    "dpad":        env_dpad,
    "button":      env_button,
    "mcq":         env_mcq,
    "touchscreen": env_touchscreen,
}

bench_results  = {}
expert_results = {}

for name, graft in all_grafts.items():
    env = envs[name]
    bench_results[name]  = BenchmarkSuite(graft, processor, device=device).run_benchmark(
        env, n_episodes=20
    )
    expert_results[name] = run_expert_baseline(env, n_episodes=10)

# Summary table
col_w = 12
header = (
    f"{'Appendage':>{col_w}s}  "
    f"{'Graft SR':>8s}  {'Graft Rew':>10s}  "
    f"{'Expert SR':>9s}  {'Expert Rew':>10s}"
)
print(header)
print("-" * len(header))
for name in all_grafts:
    b = bench_results[name]
    e = expert_results[name]
    print(
        f"{name:>{col_w}s}  "
        f"{b.success_rate:>8.2f}  {b.mean_reward:>10.2f}  "
        f"{e['success_rate']:>9.2f}  {e['mean_reward']:>10.2f}"
    )

## 6. GIF Export

Visual comparison of the expert policy and the trained joystick graft.
We also export a touchscreen rollout to compare tap accuracy.

In [ ]:
import os
os.makedirs("gifs", exist_ok=True)

# Expert rollout (no VLM required)
record_expert_gif(
    env=TargetNavEnvironment(),
    path="gifs/expert_joystick.gif",
    n_steps=60,
    seed=0,
    fps=10,
)
print("Saved: gifs/expert_joystick.gif")

# Trained joystick graft rollout
save_rollout_gif(
    graft=graft_joystick,
    processor=processor,
    env=TargetNavEnvironment(),
    path="gifs/trained_joystick.gif",
    n_steps=60,
    seed=0,
    device=device,
    fps=10,
)
print("Saved: gifs/trained_joystick.gif")

In [ ]:
# Trained touchscreen graft rollout
save_rollout_gif(
    graft=graft_touchscreen,
    processor=processor,
    env=PointingEnvironment(),
    path="gifs/trained_touchscreen.gif",
    n_steps=60,
    seed=0,
    device=device,
    fps=10,
)
print("Saved: gifs/trained_touchscreen.gif")

In [ ]:
print("Expert (joystick):")
display(IPImage(filename="gifs/expert_joystick.gif"))

print("Trained joystick graft:")
display(IPImage(filename="gifs/trained_joystick.gif"))

print("Trained touchscreen graft:")
display(IPImage(filename="gifs/trained_touchscreen.gif"))

## 7. What's Next?

Head to **`03_advanced.ipynb`** to go beyond frozen backbones:

- **LoRA fine-tuning** — adapt ~0.5 % of VLM parameters for harder tasks without full fine-tuning
- **`DEFAULT_CURRICULUM`** — the full freezing schedule that trades speed for best final performance
- **`CompositeGraft`** — attach multiple action heads to a single VLM forward pass
- **`auto_curriculum`** — one-line training with automatic environment selection
- **Save / reload / HuggingFace Hub** — share your grafts